# 멀티 벡터 검색 (Multi-Vector Retrieval) 실습

**LangChain 1.0+ | GPT-4o-mini | 네이버 뉴스 데이터셋**

---

## 💡 멀티 벡터 검색이란?

```
일반 검색의 문제:
긴 문서 전체 → 벡터화 → 검색
❌ 긴 문서는 임베딩 시 정보 손실 발생

멀티 벡터 검색의 해결:
요약/작은청크 → 벡터화 (검색용) 정확한 검색
원본/큰청크 → 저장 (반환용) 풍부한 컨텍스트

결과: 검색 정확도 ↑ + LLM 답변 품질 ↑
```

---

# 환경 설정

---

In [ ]:
# 멀티 벡터 검색(Multi-Vector Retrieval) 실습을 위한 환경 설정 코드입니다.
# 멀티 벡터 검색은 긴 문서를 처리할 때 발생하는 정보 손실 문제를 해결하기 위한 기법으로,
# 요약된 텍스트로 검색을 수행하고 원본 텍스트를 반환하는 두 단계 프로세스를 사용합니다.

# LangChain 1.0+ 및 필요한 모든 라이브러리들을 설치합니다.
# -U 옵션은 이미 설치된 패키지라도 최신 버전으로 업그레이드합니다.
# 설치할 라이브러리들:
# langchain: LangChain 핵심 프레임워크
# langchain-openai: OpenAI 모델 통합
# langchain-core: LangChain 핵심 구성 요소
# langchain-community: 커뮤니티 제공 통합 모듈
# langchain-chroma: Chroma 벡터 데이터베이스 통합
# langchain-text-splitters: 텍스트 분할 도구
# chromadb: Chroma 벡터 데이터베이스
# openai: OpenAI API 클라이언트
# datasets: Hugging Face 데이터셋 로더
# tiktoken: OpenAI 토큰화 라이브러리
!pip install -U \
    langchain \
    langchain-openai \
    langchain-core \
    langchain-community \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    openai \
    datasets \
    tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0

In [ ]:
# LangChain 버전 확인을 위해 langchain 모듈을 임포트합니다.
# 버전 확인은 코드가 LangChain 1.0+ 이상에서 정상 작동하는지 확인하기 위해 필요합니다.
import langchain

# LangChain 버전을 출력합니다.
print(f"LangChain 버전: {langchain.__version__}")

# LangChain 버전이 1.0 이상인지 확인하고 메시지를 출력합니다.
# LangChain 1.0+는 새로운 API와 기능을 제공하며, 이 실습은 해당 버전을 기준으로 작성되었습니다.
print("LangChain 1.0+ 설치 완료!" if langchain.__version__ >= "1.0" else "버전 확인 필요")

LangChain 버전: 1.1.0
LangChain 1.0+ 설치 완료!


In [ ]:
# 운영체제 상호작용을 위한 모듈 임포트
import os
# 고유 식별자 생성용 모듈 (문서 ID 생성에 사용)
import uuid
# 비밀번호 입력처럼 안전하게 사용자 입력을 받기 위한 모듈
from getpass import getpass
# Hugging Face 데이터셋 로딩을 위한 모듈
from datasets import load_dataset

# LangChain 1.0+ 모듈들 임포트
# OpenAI 임베딩 모델과 채팅 모델을 사용하기 위한 클래스
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# 메모리 내 문서 저장소 (원본 문서 저장용)
from langchain_core.stores import InMemoryStore
# Chroma 벡터 데이터베이스 통합 (요약 벡터 저장용)
from langchain_chroma import Chroma
# 표준 문서 표현 클래스
from langchain_core.documents import Document
# 대화형 프롬프트 템플릿 생성 클래스
from langchain_core.prompts import ChatPromptTemplate
# 문자열 출력 파서
from langchain_core.output_parsers import StrOutputParser
# 재귀적 문자 텍스트 분할기 (문서를 청크로 나눔)
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# OpenAI API 키 설정
# 사용자로부터 OpenAI API 키를 안전하게 입력받아 환경 변수에 저장합니다.
# os.environ을 통해 설정된 API 키는 이후 OpenAI 관련 모듈에서 자동으로 사용됩니다.
# getpass() 함수는 입력 내용이 화면에 표시되지 않도록 합니다.
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키를 입력: ")

# API 키 설정 완료 메시지 출력
print("API 키 설정 완료")

# 코드 실행 흐름 요약:
# 1. 필요한 모든 라이브러리 설치 (인터넷 연결 필요)
# 2. LangChain 버전 확인 (1.0+ 이상 필요)
# 3. 필요한 모듈 임포트
# 4. OpenAI API 키 입력 및 환경 변수 설정

# 멀티 벡터 검색의 개념:
# - 일반적인 RAG: 전체 문서를 임베딩하여 검색 → 긴 문서는 임베딩 시 정보 손실 발생
# - 멀티 벡터 검색:
#   a) 요약된 텍스트나 작은 청크로 검색 (정확한 검색)
#   b) 검색된 문서의 원본이나 큰 청크 반환 (풍부한 컨텍스트)
# - 장점: 검색 정확도 향상 + LLM 답변 품질 향상

# 이 코드가 준비하는 시스템 구조:
# 1. Vector Store: 요약 텍스트의 임베딩을 저장 (빠르고 정확한 검색용)
# 2. Doc Store: 원본 문서를 저장 (풍부한 컨텍스트 제공용)
# 3. 검색 프로세스: 사용자 쿼리 → Vector Store에서 요약 검색 → Doc Store에서 원본 문서 반환

# 다음 단계에서 수행할 작업:
# 1. 데이터셋 로드 (네이버 뉴스 요약 데이터셋)
# 2. 멀티 벡터 시스템 초기화
# 3. 문서 인덱싱 (요약 → 원본 매핑)
# 4. 멀티 벡터 검색 테스트
# 5. RAG 시스템 구축 및 테스트

OpenAI API 키를 입력: ··········
API 키 설정 완료


---

# 실습 - 요약으로 검색 → 원본 반환

---

## 📊 시스템 구조

```
┌─────────────────┐
│  사용자 쿼리    │  "경제 뉴스 찾아줘"
└────────┬────────┘
         │
         ▼
┌─────────────────────┐
│  Vector Store       │  요약으로 검색 (빠르고 정확)
│  "한은, 금리 동결"    │  
└────────┬────────────┘
         │ doc_id로 연결
         ▼
┌─────────────────────┐
│  Doc Store          │  원본 반환 (풍부한 컨텍스트)
│  "한국은행은..."      │  
└────────┬────────────┘
         │
         ▼
┌─────────────────┐
│  GPT-4o-mini    │  답변 생성
└─────────────────┘
```

## 데이터 로드

In [ ]:
# 데이터셋 로드
# Hugging Face의 데이터셋 허브에서 "daekeun-ml/naver-news-summarization-ko" 데이터셋을 불러옵니다.
# 이 데이터셋은 네이버 뉴스 기사와 그 요약문을 포함하고 있는 한국어 데이터셋입니다.
# split="train"으로 지정하여 학습 세트(기본 데이터)를 로드합니다.
dataset = load_dataset("daekeun-ml/naver-news-summarization-ko", split="train")

# 실습용 샘플 (30개)
# 전체 데이터셋 중 실습을 위해 30개의 샘플만 선택합니다.
# NUM_SAMPLES 변수에 샘플 수를 저장하여 나중에 변경이 쉽도록 합니다.
NUM_SAMPLES = 30
# dataset.select(range(NUM_SAMPLES))를 사용하여 처음부터 NUM_SAMPLES까지의 인덱스로 데이터를 선택합니다.
data = dataset.select(range(NUM_SAMPLES))

# 데이터 로드 완료 메시지 출력
# f-string을 사용하여 로드된 데이터 샘플 수를 동적으로 표시합니다.
print(f'{len(data)}개 뉴스 기사 로드 완료\n')
# 구분선 출력 (가독성 향상)
print("="*80)

# 데이터 구조 확인
# 첫 번째 샘플(인덱스 0)을 가져와서 데이터의 구조를 확인합니다.
sample = data[0]
# 샘플 데이터의 구조를 출력합니다.
print(f"[샘플 데이터 구조]")
# get() 메서드를 사용하여 'title' 필드가 없으면 'N/A'를 반환합니다.
print(f"제목: {sample.get('title', 'N/A')}")
# 요약문의 길이(문자 수)와 내용을 출력합니다.
print(f"요약: ({len(sample['summary'])}자):")
print(f"{sample.get('summary', 'N/A')}")
# 원문의 길이(문자 수)와 내용을 출력합니다.
print(f"원문: ({len(sample['document'])}자):")
print(f"{sample.get('document', 'N/A')}")

# 코드 설명:
# 1. load_dataset() 함수는 Hugging Face 데이터셋 허브에서 데이터셋을 다운로드하고 로드합니다.
#    - 첫 번째 실행 시 데이터셋을 다운로드하므로 인터넷 연결이 필요합니다.
#    - 이후에는 캐시된 데이터를 사용합니다.
# 2. 데이터셋 구조:
#    - 각 샘플은 딕셔너리 형태로, 'title', 'summary', 'document' 키를 가집니다.
#    - 'title': 뉴스 기사의 제목
#    - 'summary': 뉴스 기사의 요약문
#    - 'document': 뉴스 기사의 원문 전체
# 3. 샘플링 이유:
#    - 전체 데이터셋은 크기가 클 수 있으므로 실습 속도와 효율성을 위해 일부만 사용합니다.
#    - 30개 샘플은 멀티 벡터 검색 개념을 설명하기에 충분한 수입니다.
# 4. 데이터 구조 확인의 중요성:
#    - 데이터의 형식과 내용을 확인하여 이후 처리 단계(임베딩, 저장 등)에 적합한지 확인합니다.
#    - 예를 들어, 요약문과 원문의 길이 차이를 확인할 수 있습니다.

# 출력 예시:
# 30개 뉴스 기사 로드 완료
# ================================================================================
# [샘플 데이터 구조]
# 제목: [실제 뉴스 제목]
# 요약: (123자):
# [실제 요약문 내용]
# 원문: (1234자):
# [실제 원문 내용]

# 다음 단계:
# 이 데이터를 사용하여 멀티 벡터 검색 시스템을 구축합니다.
# 요약문은 벡터 저장소(Vector Store)에 저장되어 검색에 사용되고,
# 원문은 문서 저장소(Doc Store)에 저장되어 최종 응답 생성 시 사용됩니다.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/787 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/66.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/22194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2466 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2740 [00:00<?, ? examples/s]

30개 뉴스 기사 로드 완료

[샘플 데이터 구조]
제목: 추경호 중기 수출지원 총력 무역금융 40조 확대
요약: (170자):
올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다.
원문: (831자):
앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 

## 멀티 벡터 시스템 초기화

In [ ]:
# 임베딩 모델 초기화
# OpenAIEmbeddings 클래스를 사용하여 텍스트 임베딩 모델을 생성합니다
# model="text-embedding-3-small": OpenAI의 최신 임베딩 모델 중 하나로,
# 효율적이면서도 좋은 성능을 제공하는 소형 모델입니다
# 이 모델은 텍스트를 고차원 벡터(1536차원)로 변환하여 의미적 유사도를 계산할 수 있습니다
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 사용 중인 임베딩 모델 정보 출력
print("임베딩 모델: text-embedding-3-small")

# LLM(대형 언어 모델) 초기화
# ChatOpenAI 클래스를 사용하여 GPT-4o-mini 모델을 초기화합니다
# model="gpt-4o-mini": OpenAI의 효율적이고 저렴한 GPT-4 모델 변종
# temperature=0: 응답의 창의성을 최소화하여 일관된 결과를 생성하도록 설정
# temperature가 0이면 모델은 가장 확률이 높은 토큰을 선택하므로 결정적인 응답이 생성됩니다
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 사용 중인 LLM 정보 출력
print("LLM: gpt-4o-mini")

# Vector Store(벡터 저장소) 초기화 - 요약 문서 저장용
# Chroma 클래스를 사용하여 벡터 데이터베이스를 생성합니다
# Chroma는 오픈소스 벡터 데이터베이스로, 로컬에서 쉽게 사용할 수 있습니다
vectorstore = Chroma(
    collection_name="news_summaries",  # 컬렉션 이름 설정 (논리적 그룹화)
    embedding_function=embeddings       # 사용할 임베딩 함수 지정
)
# Vector Store 정보 출력
print("Vector Store: Chroma")

# Doc Store(문서 저장소) 초기화 - 원본 문서 저장용
# InMemoryStore 클래스를 사용하여 메모리 내 문서 저장소를 생성합니다
# 이 저장소는 원본 뉴스 기사 문서를 key-value 형태로 저장합니다
# 메모리에 저장되므로 프로그램 재시작 시 데이터가 사라집니다 (실습용으로 적합)
docstore = InMemoryStore()
# Doc Store 정보 출력
print("Doc Store: InMemoryStore")

# 코드 설명:
# 1. 임베딩 모델(embeddings):
#    - 텍스트를 수치형 벡터로 변환하는 모델
#    - "text-embedding-3-small" 모델은 1536차원 벡터를 생성하며,
#      정확도와 효율성 사이의 좋은 균형을 제공합니다
#    - 이 임베딩은 벡터 저장소에서 문서 간 유사도 검색에 사용됩니다

# 2. LLM(llm):
#    - 실제 텍스트 생성을 담당하는 모델
#    - GPT-4o-mini는 GPT-4 모델의 경량 버전으로,
#      빠른 응답 속도와 낮은 비용이 장점입니다
#    - temperature=0은 실험의 재현성을 위해 설정합니다

# 3. Vector Store(vectorstore):
#    - 요약된 텍스트의 임베딩 벡터를 저장하는 데이터베이스
#    - collection_name="news_summaries"로 요약 문서들을 하나의 그룹으로 관리
#    - embedding_function=embeddings로 위에서 정의한 임베딩 모델을 사용
#    - Chroma는 로컬 파일 시스템에 데이터를 저장하므로 외부 서버 필요 없음

# 4. Doc Store(docstore):
#    - 원본 뉴스 기사 문서를 저장하는 키-값 저장소
#    - InMemoryStore는 메모리에 데이터를 저장하는 간단한 저장소
#    - 실제 프로덕션 환경에서는 Redis, MongoDB 등 영구 저장소를 사용할 수 있음

# 멀티 벡터 검색 시스템에서의 역할:
# - Vector Store: 요약 텍스트의 임베딩을 저장 → 빠른 유사도 검색
# - Doc Store: 원본 텍스트를 저장 → 검색된 문서의 원본 내용 제공
# - 검색 과정:
#   1. 사용자 쿼리의 임베딩 생성
#   2. Vector Store에서 유사한 요약 문서 검색
#   3. 검색된 문서의 ID로 Doc Store에서 원본 문서 조회
#   4. 원본 문서를 LLM에 컨텍스트로 제공하여 응답 생성

# 출력 예상 결과:
# 임베딩 모델: text-embedding-3-small
# LLM: gpt-4o-mini
# Vector Store: Chroma
# Doc Store: InMemoryStore

# 다음 단계에서는:
# 1. 로드한 뉴스 데이터를 처리하여 요약 문서와 원본 문서를 준비
# 2. 요약 문서는 Vector Store에 임베딩과 함께 저장
# 3. 원본 문서는 Doc Store에 ID와 함께 저장
# 4. 두 저장소 간 문서 ID를 연결하여 멀티 벡터 검색 시스템 구축

임베딩 모델: text-embedding-3-small
LLM: gpt-4o-mini
Vector Store: Chroma
Doc Store: InMemoryStore


In [ ]:
# LangChain의 핵심 모듈들을 임포트합니다
from langchain_core.documents import Document  # 표준 문서 표현 클래스
from langchain_core.runnables import RunnableLambda  # 함수를 Runnable 객체로 변환하는 래퍼

# -------------------------------------------------------
# 1) parent 문서 저장 (docstore 역할)
# -------------------------------------------------------
# parent_store 변수에 docstore(InMemoryStore 객체)를 할당합니다
# 이 저장소는 원본 문서(parent 문서)를 저장하는 역할을 합니다
# 구조: {문서ID: Document객체, 문서ID2: Document객체2, ...}
# 예: docstore = { "A": Document(...), "B": Document(...) }
parent_store = docstore

# -------------------------------------------------------
# 2) MultiVectorRetriever 기능 구현
#    - child 벡터 저장소 = vectorstore
#    - parent 문서 = parent_store
#    - id_key = "doc_id"
# -------------------------------------------------------
# 멀티 벡터 검색을 수행하는 함수 정의
def multivector_retrieve(query):
    # 1. child 문서 검색 (Vector Store에서 요약 문서 검색)
    # vectorstore.similarity_search() 메서드를 사용하여 쿼리와 유사한 요약 문서 검색
    # k=5: 상위 5개의 유사한 문서 반환
    child_docs = vectorstore.similarity_search(query, k=5)

    # 2. child -> parent id 매핑
    # 검색된 child 문서들의 메타데이터에서 "doc_id"를 추출하여 집합(set)으로 저장
    # 집합을 사용하여 중복 ID 제거
    parent_ids = {doc.metadata["doc_id"] for doc in child_docs}

    # 3. parent 문서 반환
    # 집합의 각 ID에 대해 parent_store에서 해당 문서를 조회
    # 리스트 컴프리헨션을 사용하여 문서 조회
    # 주의: 원래 코드에 오타가 있습니다 - "in id in parent_store"는 잘못된 구문입니다
    # 올바른 코드: [parent_store[id] for id in parent_ids if id in parent_store]
    # 하지만 InMemoryStore 객체는 인덱싱([id])이 아닌 mget() 메서드를 사용해야 합니다
    parent_docs = [parent_store[id] for id in parent_ids in id in parent_store]  # 이 줄에는 오타가 있음 수정 필요: [parent_store.mget([id])[0] for id in parent_ids]
    return parent_docs

# LCEL Runnable 형태의 retriever 만들기 (MultiVectorRetriever 대체)
# RunnableLambda를 사용하여 multivector_retrieve 함수를 Runnable 객체로 변환합니다
# 이렇게 하면 LangChain 체인에서 retriever를 다른 컴포넌트와 연결할 수 있습니다
retriever = RunnableLambda(multivector_retrieve)

print("Custom MultiVectorRetriever 생성 완료")
print("="*80)

# 코드의 문제점 및 수정 필요 사항:
# 1. 16번 줄의 리스트 컴프리헨션에 구문 오류가 있습니다:
#    원래: [parent_store[id] for id in parent_ids in id in parent_store]
#    수정 필요: [parent_store.mget([id])[0] for id in parent_ids]
#    또는: parent_store.mget(list(parent_ids))
#
# 2. InMemoryStore는 인덱싱([id])을 지원하지 않습니다. mget() 메서드를 사용해야 합니다:
#    parent_store.mget(list(parent_ids))는 ID 리스트를 받아 문서 리스트를 반환합니다
#
# 3. 실제 구현에서는 다음과 같이 수정해야 합니다:
#    parent_ids_list = list(parent_ids)
#    parent_docs = parent_store.mget(parent_ids_list)
#    parent_docs = [doc for doc in parent_docs if doc is not None]  # None 필터링

# 멀티 벡터 검색의 작동 원리:
# 1. 사용자 쿼리(query)가 입력됩니다
# 2. Vector Store(요약 저장소)에서 쿼리와 유사한 요약 문서(child docs)를 검색합니다
# 3. 검색된 요약 문서의 메타데이터에서 doc_id를 추출합니다
# 4. 추출된 doc_id들을 사용하여 Doc Store(원본 저장소)에서 원본 문서(parent docs)를 조회합니다
# 5. 원본 문서들을 반환합니다

# 이 접근법의 장점:
# - 검색 정확도: 짧은 요약문으로 검색하면 더 정확한 유사도 계산이 가능합니다
# - 컨텍스트 풍부함: 반환되는 원본 문서는 LLM에 풍부한 컨텍스트를 제공합니다
# - 효율성: 요약문은 짧아 임베딩 생성과 검색이 빠릅니다

# 실제 작동 예시:
# 쿼리: "경제 성장률 관련 뉴스"
# 1. Vector Store에서 "경제", "성장률"과 관련된 요약 문서 5개 검색
# 2. 검색된 요약 문서들의 doc_id 추출 (예: ["id1", "id3", "id7"])
# 3. Doc Store에서 id1, id3, id7에 해당하는 원본 문서 조회
# 4. 원본 문서들 반환

# 다음 단계에서 이 retriever를 실제로 사용하기 전에,
# 먼저 Vector Store와 Doc Store에 데이터를 적절히 저장해야 합니다.
# 저장 시 동일한 문서에 대해 요약문(Vector Store)과 원본문(Doc Store)이 같은 doc_id를 공유하도록 해야 합니다.

Custom MultiVectorRetriever 생성 완료


## 문서 인덱싱 (요약 → 원본 매핑)

In [ ]:
# 고유 식별자 생성을 위한 uuid 모듈 임포트
import uuid
# LangChain의 표준 문서 표현 클래스 임포트
from langchain_core.documents import Document

# 문서 인덱싱을 위한 빈 리스트 초기화
# doc_ids: 각 문서에 할당된 고유 ID 저장
doc_ids = []
# summary_docs: Vector Store에 저장할 요약 문서들 저장
summary_docs = []
# full_docs: Doc Store에 저장할 원본 문서들 저장
full_docs = []

# data 리스트의 각 항목(뉴스 기사)에 대해 반복 처리
# enumerate를 사용하여 인덱스(idx)와 항목(item)을 동시에 가져옵니다
for idx, item in enumerate(data):
    # 고유 ID 생성
    # uuid.uuid4()는 무작위 UUID(Universally Unique Identifier)를 생성합니다
    # str()을 사용하여 문자열로 변환합니다 (예: "123e4567-e89b-12d3-a456-426614174000")
    doc_id = str(uuid.uuid4())
    # 생성된 ID를 doc_ids 리스트에 추가
    doc_ids.append(doc_id)

    # -------------------------
    # 1) 원본 문서 (DocStore 저장용) 생성
    # -------------------------
    # Document 객체를 생성하여 원본 뉴스 기사를 저장합니다
    full_doc = Document(
        page_content=item['document'],  # 문서 내용: 원본 뉴스 기사
        metadata={
            "doc_id": doc_id,  # 문서 고유 ID (요약 문서와 동일한 ID 사용)
            "title": item.get('title', f'뉴스_{idx}'),  # 뉴스 제목 (없으면 기본값 사용)
            "type": "full_document"  # 문서 유형 표시 (원본 문서임을 나타냄)
        }
    )
    # 생성된 원본 문서를 full_docs 리스트에 추가
    full_docs.append(full_doc)

    # -------------------------
    # 2) 요약 문서 (VectorStore 검색용) 생성
    # -------------------------
    # Document 객체를 생성하여 요약 뉴스 기사를 저장합니다
    summary_doc = Document(
        page_content=item['summary'],  # 문서 내용: 뉴스 요약문
        metadata={"doc_id": doc_id}  # 문서 고유 ID (원본 문서와 동일한 ID 사용)
        # 요약 문서에는 최소한의 메타데이터만 저장하여 검색 효율성을 높입니다
    )
    # 생성된 요약 문서를 summary_docs 리스트에 추가
    summary_docs.append(summary_doc)

# 인덱싱 과정 시작 알림 출력
print("\n 문서 인덱싱")
# 구분선 출력 (가독성 향상)
print("="*80)

# 코드 설명:
# 1. uuid를 사용한 고유 ID 생성:
#    - 각 뉴스 기사마다 전 세계에서 유일한 ID를 부여합니다
#    - 이를 통해 원본 문서와 요약 문서를 연결할 수 있습니다
#    - UUID는 128비트 숫자로, 중복 가능성이 극히 낮습니다

# 2. 원본 문서(full_doc) 생성:
#    - page_content: 원본 뉴스 기사 전체 텍스트
#    - metadata:
#        * doc_id: 요약 문서와 연결하기 위한 동일한 ID
#        * title: 뉴스 제목 (검색 결과 표시에 사용)
#        * type: "full_document"로 표시하여 원본 문서임을 식별

# 3. 요약 문서(summary_doc) 생성:
#    - page_content: 뉴스 요약문 (원본보다 짧은 텍스트)
#    - metadata: doc_id만 포함하여 원본 문서와 연결
#    - 요약문이 짧기 때문에 임베딩 생성과 유사도 검색이 더 빠르고 정확합니다

# 4. 멀티 벡터 검색을 위한 연결 메커니즘:
#    - 동일한 doc_id를 사용하여 원본 문서와 요약 문서를 연결합니다
#    - 검색 과정:
#        a. Vector Store에서 요약 문서를 검색 (빠르고 정확)
#        b. 검색된 요약 문서의 doc_id를 추출
#        c. Doc Store에서 해당 doc_id를 가진 원본 문서를 조회
#        d. 원본 문서를 LLM에 컨텍스트로 제공

# 5. 데이터 흐름:
#    원본 데이터 → 요약문(Vector Store) + 원본문(Doc Store) → 동일 doc_id로 연결

# 출력 예시:
#
#  문서 인덱싱
# ================================================================================

# 다음 단계:
# 이제 생성된 문서들을 각각의 저장소에 저장해야 합니다:
# 1. summary_docs → Vector Store (Chroma)에 임베딩과 함께 저장
# 2. full_docs → Doc Store (InMemoryStore)에 ID와 함께 저장
# 3. 저장이 완료되면 멀티 벡터 검색이 가능해집니다

# 주의사항:
# - 요약문은 검색용으로, 원본문은 응답 생성용으로 사용됩니다
# - 동일한 doc_id를 공유하므로 두 저장소 간 문서 매핑이 가능합니다
# - 실제 프로덕션 환경에서는 데이터베이스나 분산 저장소를 사용할 수 있습니다


 문서 인덱싱


In [ ]:
# ---------------------------------------------------------
# STEP 1: Vector Store에 요약 문서 추가
# ---------------------------------------------------------
# Vector Store(Chroma)에 요약 문서들을 추가하는 단계입니다
# 요약 문서들은 검색에 사용될 것이며, 각 문서는 임베딩으로 변환되어 저장됩니다
print("Step 1: Vector Store에 요약 벡터 추가")
# vectorstore.add_documents() 메서드를 사용하여 요약 문서 리스트를 벡터 저장소에 추가합니다
# 이 메서드는 내부적으로 다음 작업을 수행합니다:
# 1. 각 요약 문서를 text-embedding-3-small 모델을 사용하여 벡터로 변환
# 2. 변환된 벡터와 문서 내용을 Chroma 데이터베이스에 저장
# 3. 문서의 메타데이터(여기서는 doc_id)도 함께 저장
vectorstore.add_documents(summary_docs)

# ---------------------------------------------------------
# STEP 2: DocStore에 원본 문서 저장
# ---------------------------------------------------------
# Doc Store(InMemoryStore)에 원본 문서들을 저장하는 단계입니다
# 원본 문서들은 나중에 검색된 문서의 전체 내용을 제공하는 데 사용됩니다
print("Step 2: Doc Store에 원본 문서 저장")
# docstore.mset() 메서드는 여러 키-값 쌍을 한 번에 저장합니다
# list(zip(doc_ids, full_docs))는 다음과 같은 리스트를 생성합니다:
# [(doc_id1, full_doc1), (doc_id2, full_doc2), ...]
# 이렇게 하면 각 문서 ID가 키가 되고, 원본 Document 객체가 값이 됩니다
docstore.mset(list(zip(doc_ids, full_docs)))

# 저장 완료 메시지 출력
print("="*80)
print("인덱싱 완료! 검색 가능")
print("="*80)

# 코드 설명:
# 1. Vector Store 저장 과정:
#    - summary_docs 리스트에는 30개의 요약 Document 객체가 들어 있습니다
#    - 각 문서는 page_content(요약 텍스트)와 metadata(doc_id)를 포함합니다
#    - add_documents() 호출 시 자동으로 임베딩이 생성되고 Chroma에 저장됩니다
#    - Chroma는 로컬에 데이터를 저장하며, 기본적으로 ./chroma_db/ 디렉토리에 저장됩니다

# 2. Doc Store 저장 과정:
#    - zip(doc_ids, full_docs)는 두 리스트의 요소를 짝지어 튜플로 만듭니다
#    - 예: [(id1, doc1), (id2, doc2), ..., (id30, doc30)]
#    - mset()은 이 키-값 쌍들을 InMemoryStore에 저장합니다
#    - InMemoryStore는 메모리 내 딕셔너리로, {id1: doc1, id2: doc2, ...} 형식으로 저장됩니다

# 3. 두 저장소 간의 관계:
#    - Vector Store: 요약 문서 + 임베딩 벡터 저장 (검색용)
#    - Doc Store: 원본 문서 저장 (내용 제공용)
#    - 연결 고리: doc_id (두 저장소에 동일한 ID로 저장됨)

# 4. 저장 후 시스템 상태:
#    - Vector Store(Chroma): 30개의 요약 문서 임베딩이 색인됨
#    - Doc Store(InMemoryStore): 30개의 원본 문서가 메모리에 저장됨
#    - 이제 멀티 벡터 검색이 가능한 상태가 됨

# 출력 예상:
# Step 1: Vector Store에 요약 벡터 추가
# Step 2: Doc Store에 원본 문서 저장
# ================================================================================
# 인덱싱 완료! 검색 가능
# ================================================================================

# 주의사항:
# 1. Vector Store 저장 시 임베딩 생성에 시간이 걸릴 수 있습니다 (OpenAI API 호출)
# 2. 많은 문서를 처리할 경우 API 비용과 시간이 증가할 수 있습니다
# 3. Doc Store는 메모리 저장소이므로 프로그램 재시작 시 데이터가 사라집니다
#    - 실제 애플리케이션에서는 Redis, PostgreSQL 등의 영구 저장소를 사용합니다

# 다음 단계:
# 이제 멀티 벡터 검색을 테스트할 수 있습니다:
# 1. 사용자 쿼리를 Vector Store에서 검색 (요약 문서 기반)
# 2. 검색된 문서의 doc_id로 Doc Store에서 원본 문서 조회
# 3. 원본 문서를 LLM에 컨텍스트로 제공하여 답변 생성

# 멀티 벡터 검색의 장점이 여기서 실현됩니다:
# - 요약문으로 검색: 빠르고 정확한 유사도 계산
# - 원본문으로 응답: 풍부한 컨텍스트 제공
# - 분리된 저장: 각 저장소에 최적화된 형태로 데이터 저장

Step 1: Vector Store에 요약 벡터 추가
Step 2: Doc Store에 원본 문서 저장
인덱싱 완료! 검색 가능


## 멀티 벡터 검색 테스트

In [ ]:
# LangChain의 RunnableLambda를 임포트하여 함수를 Runnable 객체로 변환할 수 있게 합니다
from langchain_core.runnables import RunnableLambda

# 멀티 벡터 검색을 수행하는 함수를 정의합니다
# 이 함수는 딕셔너리 형태의 파라미터를 받아서 처리합니다
def multivector_retrieve(params: dict):
    # 파라미터에서 검색 쿼리를 추출합니다
    query = params["query"]
    # 파라미터에서 k값(반환할 문서 수)을 추출하며, 기본값은 3으로 설정합니다
    k = params.get("k", 3)

    # 1단계: 요약(Child) 문서 검색
    # vectorstore(Chroma)에서 similarity_search 메서드를 사용하여
    # 쿼리와 유사한 요약 문서를 검색합니다
    # k개의 상위 문서를 반환받습니다
    child_docs = vectorstore.similarity_search(query, k=k)

    # 2단계: child 문서에서 parent 문서 ID 매핑
    # 검색된 child 문서들의 메타데이터에서 "doc_id"를 추출하여 집합(set)으로 저장합니다
    # 집합을 사용하여 중복 ID를 자동으로 제거합니다
    parent_ids = {d.metadata["doc_id"] for d in child_docs}
    # 집합을 리스트로 변환합니다 (mget 메서드에 리스트 형태로 전달하기 위해)
    parent_ids_list = list(parent_ids)

    # 3단계: parent 문서 조회
    # docstore(InMemoryStore)의 mget 메서드를 사용하여
    # ID 리스트에 해당하는 원본 문서들을 한 번에 조회합니다
    parent_docs = docstore.mget(parent_ids_list)
    # 조회 결과에서 None 값(존재하지 않는 ID에 대한 결과)을 필터링합니다
    parent_docs = [d for d in parent_docs if d is not None]

    # 최종적으로 원본 문서들을 반환합니다
    return parent_docs

# multivector_retrieve 함수를 RunnableLambda로 감싸서 LangChain runnable 객체를 생성합니다
# 이를 통해 이 함수를 LangChain 체인의 일부로 사용할 수 있습니다
retriever = RunnableLambda(multivector_retrieve)

# 준비 완료 메시지 출력
print("Custom Multi-Vector Retrieve 준비 완료")

# 코드 설명:
# 1. 함수 시그니처: params: dict
#    - 함수가 딕셔너리를 입력받도록 설계되었습니다
#    - 이는 LangChain 체인에서 편리하게 사용하기 위한 설계입니다
#    - params에는 "query"(검색어)와 "k"(반환할 문서 수)가 포함됩니다

# 2. similarity_search 메서드:
#    - Chroma 벡터 저장소에서 제공하는 메서드로, 쿼리와 유사한 문서를 검색합니다
#    - 내부적으로 쿼리를 임베딩으로 변환하고, 저장된 문서 임베딩들과 코사인 유사도를 계산합니다
#    - 상위 k개의 가장 유사한 문서를 반환합니다

# 3. 집합(set) 사용의 장점:
#    - 여러 child 문서가 동일한 parent 문서를 가리킬 수 있습니다
#    - 집합을 사용하면 중복 ID를 자동으로 제거하여 효율적인 조회가 가능합니다

# 4. mget 메서드:
#    - InMemoryStore의 메서드로, 여러 키에 대한 값을 한 번에 조회합니다
#    - 이는 네트워크나 데이터베이스 호출을 최소화하기 위한 최적화입니다

# 5. None 필터링:
#    - 어떤 이유로든 저장소에 존재하지 않는 ID가 있을 수 있습니다
#    - 이러한 경우를 대비하여 None 값을 필터링합니다

# 멀티 벡터 검색의 전체 프로세스:
# 1. 사용자 쿼리를 받습니다
# 2. Vector Store에서 요약 문서를 검색합니다 (빠르고 정확)
# 3. 검색된 요약 문서들의 doc_id를 수집합니다
# 4. 수집된 doc_id들로 Doc Store에서 원본 문서를 조회합니다
# 5. 원본 문서들을 반환합니다 (LLM에 풍부한 컨텍스트 제공)

# 이 retriever의 특징:
# - 검색은 요약문으로 수행하지만, 반환은 원본문으로 합니다
# - 요약문은 짧아서 임베딩 품질이 좋고 검색이 빠릅니다
# - 원본문은 길어서 LLM에 더 많은 정보를 제공할 수 있습니다

# 사용 예시:
# retriever.invoke({"query": "경제 성장률", "k": 3})
# → 쿼리 "경제 성장률"과 유사한 요약문을 가진 문서 3개의 원본문을 반환

# 다음 단계에서는 이 retriever를 사용하여 실제 검색을 테스트하고,
# RAG 시스템에서 사용할 것입니다.

Custom Multi-Vector Retrieve 준비 완료


In [ ]:
# 문서 검색을 위한 사용자 친화적인 함수를 정의합니다
def search_documents(query, k=3):
    # 검색 쿼리 출력 (사용자가 무엇을 검색하는지 확인)
    print(f"검색 쿼리: {query}")
    # 구분선 출력 (가독성 향상)
    print("="*80)

    # retriever를 사용하여 문서 검색 실행
    # retriever.invoke() 메서드는 딕셔너리 형태의 파라미터를 기대합니다
    # {"query": query, "k": k} 형식으로 전달합니다
    results = retriever.invoke({"query":query, "k":k})

    # 검색 결과 개수 출력
    print(f"검색 결과: {len(results)}개 문서")

    # 검색된 각 문서에 대해 상세 정보 출력
    # enumerate를 사용하여 문서 번호(1부터 시작)와 문서 객체를 함께 가져옵니다
    for i, doc in enumerate(results, 1):
        # 문서 구분선과 번호 출력
        print(f"\n[문서 {i}]")
        # 문서 제목 출력 (없으면 'N/A', 최대 70자로 제한)
        print(f"제목: {doc.metadata.get('title', 'N/A')[:70]}")
        # 문서 타입 출력 (원본 문서인지 확인)
        print(f"타입: {doc.metadata.get('type', 'N/A')}")
        # 문서 길이(문자 수) 출력
        print(f"길이: {len(doc.page_content)}자")
        # 문서 내용 미리보기 출력
        print(f"내용 미리보기:")
        # 문서 내용의 처음 200자만 출력하고 말줄임표 추가
        print(f"{doc.page_content[:200]}...")

    # 검색 결과 출력 종료를 위한 구분선
    print("="*80)
    # 검색 결과를 반환 (다른 함수에서 사용할 수 있도록)
    return results

# 함수 정의 완료 메시지
print("검색 함수 준비 완료")

# 코드 설명:
# 1. 함수 정의: search_documents(query, k=3)
#    - query: 사용자의 검색어 (필수)
#    - k: 반환할 문서 수 (선택, 기본값 3)
#    - 반환값: 검색된 Document 객체들의 리스트

# 2. retriever.invoke() 호출:
#    - 이전에 정의한 multivector_retrieve 함수가 실행됩니다
#    - 내부적으로:
#        a. Vector Store에서 요약 문서 검색
#        b. 검색된 요약 문서의 doc_id 수집
#        c. Doc Store에서 해당 doc_id의 원본 문서 조회
#        d. 원본 문서들 반환

# 3. 결과 출력의 세부사항:
#    - 제목은 최대 70자로 제한: 너무 긴 제목은 잘라서 표시
#    - 내용 미리보기는 200자로 제한: 전체 내용이 너무 길 경우 요약 표시
#    - metadata.get() 메서드: 해당 키가 없으면 기본값('N/A') 반환

# 4. 반환값의 활용:
#    - 함수는 검색 결과를 반환하므로, 다른 함수에서 이 결과를 사용할 수 있습니다
#    - 예: RAG 시스템에서 검색 결과를 LLM에 전달할 수 있습니다

# 5. 사용자 경험 개선:
#    - 명확한 출력 형식으로 사용자가 검색 결과를 쉽게 이해할 수 있도록 합니다
#    - 각 문서의 핵심 정보(제목, 길이, 내용 일부)를 제공합니다

# 예상 출력 형식:
# 검색 쿼리: 경제 성장률
# ================================================================================
# 검색 결과: 3개 문서
#
# [문서 1]
# 제목: 한국은행, 기준금리 동결 결정...경제 성장률 전망 하향
# 타입: full_document
# 길이: 1250자
# 내용 미리보기:
# 한국은행은 오늘 금융통화위원회를 열고 기준금리를 3.5%로 동결하기로 결정했다. 이번 결정은...
#
# [문서 2]
# 제목: 2분기 경제성장률 0.6%로 둔화...수출 부진 영향
# 타입: full_document
# 길이: 980자
# 내용 미리보기:
# 한국은행이 발표한 잠정집계에 따르면 2분기 실질 국내총생산(GDP)은 전분기 대비 0.6%...
#
# [문서 3]
# 제목: 정부, 올해 경제성장률 목표 2.4% 유지...지원책 강화
# 타입: full_document
# 길이: 1100자
# 내용 미리보기:
# 정부는 20일 열린 경제관계장관회의에서 올해 경제성장률 목표를 기존과 같은 2.4%로...
# ================================================================================

# 다음 단계에서는 이 함수를 사용하여 실제 검색을 테스트할 것입니다:
# 1. 경제 관련 검색 테스트
# 2. 기술 관련 검색 테스트
# 3. 검색 결과를 기반으로 RAG 시스템 구축

# 주의사항:
# - 검색 결과는 요약문의 유사도에 기반하므로, 요약문이 원문을 잘 대표해야 정확한 검색이 가능합니다
# - k 값은 사용자의 요구와 시스템 성능에 따라 조정할 수 있습니다

검색 함수 준비 완료


In [ ]:
# 테스트 1: 경제 관련 검색
# 쿼리: "경제 성장률과 금리 정책"
# 검색된 문서 수: 3개
results1 = search_documents("경제 성장률과 금리 정책", k=3)

# 출력 결과 분석 - 테스트 1:
# 검색 쿼리: 경제 성장률과 금리 정책
# ================================================================================
# 검색 결과: 3개 문서
#
# [문서 1]
# 제목: 추경호 중기 수출지원 총력 무역금융 40조 확대
# 타입: full_document
# 길이: 831자
# 내용 미리보기:
# 앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다...
#
# [문서 2]
# 제목: 글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망
# 타입: full_document
# 길이: 1796자
# 내용 미리보기:
# 경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다...
#
# [문서 3]
# 제목: 전자·철강·석유 업종 하반기 수출감소 불가피
# 타입: full_document
# 길이: 1303자
# 내용 미리보기:
# 원자재 가격 상승에 따른 수출경쟁력 약화·공급망 애로 타격 글로벌 원자재 수급난...

검색 쿼리: 경제 성장률과 금리 정책
검색 결과: 3개 문서

[문서 1]
제목: 추경호 중기 수출지원 총력 무역금융 40조 확대
타입: full_document
길이: 831자
내용 미리보기:
앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 ...

[문서 2]
제목: 글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망
타입: full_document
길이: 1796자
내용 미리보기:
경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가...

[문서 3]
제목: 전자·철강·석유 업종 하반기 수출감소 불가피
타입: full_document
길이: 1303자
내용 미리보기:
원자재 가격 상승에 따른 수출경쟁력 약화·공급망 애로 타격 글로벌 원자재 수급난 및 공급망 차질로 인해 올해 하반기 우리나라 수출 증가세가 크게 꺾일 것이라는 전망이 제기됐다. 1일 전국경제인연합회가 시장조사 전문기관 모노리서치에 의뢰해 매출액 1000대 기업 중 12대 수출 주력 업종을 대상으로 2022 하반기 수출 전망 조사 를 진행한 결과 기업들은 평...


In [ ]:
# 테스트 2: 기술 관련 검색
# 쿼리: "인공지능과 기술 혁신"
# 검색된 문서 수: 3개
results2 = search_documents("인공지능과 기술 혁신", k=3)

# 출력 결과 분석 - 테스트 2:
# 검색 쿼리: 인공지능과 기술 혁신
# ================================================================================
# 검색 결과: 3개 문서
#
# [문서 1]
# 제목: 야놀자 포커스미디어와 '동네가게 오래함께' 캠페인 진행
# 타입: full_document
# 길이: 522자
# 내용 미리보기:
# 사진 야놀자 제공 야놀자가 포커스미디어와 동네가게 오래함께 캠페인을 진행한다고...
#
# [문서 2]
# 제목: 과기정통부 7월 한 달간 제11회 정보보호의 달 운영
# 타입: full_document
# 길이: 1512자
# 내용 미리보기:
# 국정원·행안부·국방부와 협력해 다양한 온·오프라인 행사 진행 과학기술정보통신부와...
#
# [문서 3]
# 제목: SK바사 해외 사업 조직 개편... 글로벌 탑티어 기업으로 성장 박차
# 타입: full_document
# 길이: 983자
# 내용 미리보기:
# SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다...

# 검색 결과 분석:

# 테스트 1 (경제 관련 검색) 결과 평가:
# 1. 관련성: "경제 성장률과 금리 정책" 쿼리에 대해 경제 관련 문서들이 검색되었습니다.
#    - 문서1: 정부의 수출 지원 정책 (경제 정책)
#    - 문서2: 글로벌 스마트폰 시장 전망 (경제 동향)
#    - 문서3: 수출 업종 전망 (경제 예측)
# 2. 정확도: 세 문서 모두 경제 관련 주제를 다루고 있으나, "금리 정책"에 직접적으로 관련된 문서는 없습니다.
# 3. 원인 분석:
#    - 데이터셋의 제한: 30개 샘플만 사용했으므로 특정 주제(금리)에 대한 문서가 없을 수 있습니다.
#    - 요약문의 품질: 요약문이 원문의 모든 주제를 완벽히 반영하지 못할 수 있습니다.

# 테스트 2 (기술 관련 검색) 결과 평가:
# 1. 관련성: "인공지능과 기술 혁신" 쿼리에 대한 결과가 기대에 미치지 못합니다.
#    - 문서1: 마케팅 캠페인 (기술 혁신과 직접적 관련 없음)
#    - 문서2: 정보보호 캠페인 (일부 기술 관련성 있음)
#    - 문서3: 기업 조직 개편 (기술 혁신과 약간의 관련성)
# 2. 문제점:
#    - 데이터셋에 인공지능 관련 문서가 부족할 수 있습니다.
#    - "인공지능"이라는 키워드가 요약문에 포함되지 않았을 수 있습니다.
#    - 벡터 임베딩의 의미적 유사도가 예상과 다를 수 있습니다.

# 멀티 벡터 검색 시스템의 성능 평가:
# 1. 장점:
#    - 시스템이 정상적으로 작동하여 문서를 검색하고 반환합니다.
#    - 요약문으로 검색하고 원본문을 반환하는 메커니즘이 작동합니다.
#    - 검색 속도와 결과 출력이 원활합니다.
# 2. 개선 필요 사항:
#    - 데이터셋의 크기와 다양성 부족
#    - 특정 주제에 대한 문서 부재
#    - 검색 정확도 향상 필요

# 데이터셋의 한계:
# - 30개 샘플로는 다양한 주제를 커버하기 어렵습니다.
# - 실제 응용에서는 수천, 수만 개의 문서가 필요합니다.
# - 데이터의 최신성도 중요한 요소입니다.

# 다음 단계에서의 개선 방안:
# 1. 더 큰 데이터셋 사용
# 2. 다양한 주제의 문서 포함
# 3. 검색 알고리즘 튜닝 (k값 조정, 가중치 조정 등)
# 4. 하이브리드 검색 도입 (키워드 + 의미 검색)

# 실제 RAG 시스템에서의 활용:
# 이 검색 결과들을 LLM에 컨텍스트로 제공하면:
# 1. 테스트 1의 결과: 경제 정책과 동향에 대한 답변 생성 가능
# 2. 테스트 2의 결과: 인공지능에 대한 직접적 정보는 부족하지만,
#    기술 관련 일반적인 정보는 제공 가능

# 주의사항:
# - 검색 결과의 품질은 데이터의 양과 질에 크게 의존합니다.
# - 멀티 벡터 검색은 요약문의 품질이 중요합니다.
# - 실제 프로덕션 환경에서는 더 많은 데이터와 정교한 튜닝이 필요합니다.

# 다음으로는 이 검색 결과를 바탕으로 RAG 시스템을 구축하여
# LLM이 검색된 문서를 참조하여 답변을 생성하도록 할 것입니다.

검색 쿼리: 인공지능과 기술 혁신
검색 결과: 3개 문서

[문서 1]
제목: 야놀자 포커스미디어와 ‘동네가게 오래함께’ 캠페인 진행
타입: full_document
길이: 522자
내용 미리보기:
사진 야놀자 제공 야놀자가 포커스미디어와 동네가게 오래함께 캠페인을 진행한다고 4일 밝혔다. 이번 캠페인은 지역 내 우수 소상공인을 발굴하고 이들을 위한 맞춤형 광고를 제작해 해당 지역 내 홍보를 지원한다. 총 14억 원 규모의 광고 제작 및 송출 비용은 양사가 전액 부담한다. 야놀자는 제휴점을 대상으로 사연을 공모하고 상권 빅데이터 분석을 진행해 지원 대...

[문서 2]
제목: 과기정통부 7월 한 달간 제11회 정보보호의 달 운영
타입: full_document
길이: 1512자
내용 미리보기:
국정원·행안부·국방부와 협력해 다양한 온·오프라인 행사 진행 과학기술정보통신부와 한국정보보호산업협회 KISIA 은 튼튼한 사이버안보 안전한 디지털강국 를 주제로 7월 한 달 동안 제11회 정보보호의 달을 운영한다고 3일 밝혔다. 정보보호의 달은 증가하는 사이버위협에 대응하여 국민들의 보안 인식을 제고하고 정보보호 실천문화를 확산시키기 위해 매년 7월 운영되...

[문서 3]
제목: SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차
타입: full_document
길이: 983자
내용 미리보기:
SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사이언스는 기존 해외사업개발실을 BD Business Development 1 3실로 확대 재편하고 글로벌 규제 및 허가 전담 조직인 Global RA Regulatory Affairs 실을 신설한다고 1일 밝혔다. SK바이오사이언스는 지난해 코로나19 COVID 19 백신 위...


## RAG 시스템 구축

In [ ]:
# RAG(Retrieval-Augmented Generation) 시스템을 위한 프롬프트 템플릿 생성
# ChatPromptTemplate.from_template() 메서드를 사용하여 템플릿 문자열에서 프롬프트 템플릿 객체를 생성합니다
rag_prompt = ChatPromptTemplate.from_template(
    """당신은 한국 뉴스 분석 전문가입니다. 주어진 뉴스 기사를 바탕으로 질문에 답변하세요.

검색된 뉴스 기사:
{context}

질문: {question}

답변 (검색된 기사 내용을 바탕으로 정확하게 답변해주세요):"""
)

# RAG 체인 구성
# LangChain Expression Language(LCEL)를 사용하여 프롬프트 → LLM → 출력 파서 순서로 체인을 구성합니다
# '|' 연산자는 각 컴포넌트를 연결하는 파이프라인을 만듭니다
rag_chain = rag_prompt | llm | StrOutputParser()

# 시스템 준비 완료 메시지 출력
print("RAG 시스템 준비")

# 코드 설명:

# 1. RAG 프롬프트 템플릿:
#    - 역할 설정: "당신은 한국 뉴스 분석 전문가입니다." - LLM에게 특정 역할을 부여하여 응답 품질을 향상시킵니다
#    - 지시사항: "주어진 뉴스 기사를 바탕으로 질문에 답변하세요." - 외부 지식을 참조하도록 지시합니다
#    - 변수:
#        * {context}: 검색된 문서들이 삽입될 자리 (여러 문서의 내용이 결합되어 들어갑니다)
#        * {question}: 사용자의 질문이 삽입될 자리
#    - 추가 지시: "검색된 기사 내용을 바탕으로 정확하게 답변해주세요" - 환각(hallucination)을 줄이고 정확한 답변을 유도합니다

# 2. RAG 체인의 구성 요소:
#    - rag_prompt: ChatPromptTemplate 객체 - 템플릿에 context와 question 변수를 채워 완성된 프롬프트 생성
#    - llm: ChatOpenAI 객체 (GPT-4o-mini) - 프롬프트를 처리하여 응답 생성
#    - StrOutputParser(): 출력 파서 - LLM의 응답에서 텍스트만 추출하여 반환

# 3. 체인의 작동 원리:
#    - rag_chain.invoke({"context": context_text, "question": query}) 형식으로 호출
#    - 내부 처리:
#        1. rag_prompt가 context와 question을 템플릿에 삽입하여 완성된 프롬프트 생성
#        2. 완성된 프롬프트를 llm에 전달하여 OpenAI API 호출
#        3. LLM의 응답을 StrOutputParser()가 문자열로 변환
#        4. 최종 문자열 응답 반환

# 4. RAG 시스템의 전체적인 데이터 흐름:
#    사용자 질문 → 검색기(retriever) → 관련 문서 검색 → 문서들을 context로 변환 →
#    RAG 체인 → LLM 응답 → 사용자에게 최종 답변

# 5. 프롬프트 엔지니어링의 중요성:
#    - 역할 부여: "한국 뉴스 분석 전문가"로 설정하여 도메인 특화된 응답 유도
#    - 명확한 지시: "검색된 기사 내용을 바탕으로"라고 명시하여 외부 정보 활용 강조
#    - 정확성 요구: "정확하게 답변해주세요"로 환각 방지

# 6. 컨텍스트(context)의 형식:
#    - 일반적으로 여러 문서의 내용을 결합하여 하나의 문자열로 만듭니다
#    - 예시 형식:
#        """
#        [문서 1 제목]
#        문서 1 내용...
#
#        [문서 2 제목]
#        문서 2 내용...
#
#        [문서 3 제목]
#        문서 3 내용...
#        """

# 7. 이 RAG 시스템의 특징:
#    - 검색 증강: 외부 지식(뉴스 기사)을 활용하여 LLM의 지식 한계를 보완
#    - 최신 정보: 뉴스 데이터셋을 사용하므로 최신 정보 제공 가능
#    - 출처 기반: 검색된 문서를 기반으로 하므로 응답의 신뢰성 향상

# 다음 단계에서는 이 RAG 체인을 실제로 사용하는 함수를 정의하고,
# 검색기(retriever)와 연동하여 완전한 RAG 시스템을 구동할 것입니다.

RAG 시스템 준비


In [ ]:
# RAG 질의응답 함수를 정의합니다
# 이 함수는 사용자의 질문을 받아 관련 문서를 검색하고, LLM을 사용하여 답변을 생성합니다
def ask_question(question, k=3):
    # 질문 출력 (사용자가 무엇을 질문했는지 표시)
    # 주의: 원래 코드에 오타가 있습니다 - f-string을 사용해야 합니다
    print("질문: {question}")  # 오타: f-string이 아니므로 {question}이 그대로 출력됨
    # 올바른 코드: print(f"질문: {question}")
    print("="*80)

    # 문서 검색
    # retriever를 사용하여 질문과 관련된 문서를 검색합니다
    # retriever.invoke()는 {"query": question, "k": k} 형식의 딕셔너리를 받습니다
    docs = retriever.invoke({"query": question, "k":k})
    print(f"검색 완료")

    # 검색 결과가 없는 경우 처리
    if not docs:
        print("관련 문서를 찾지 못했습니다")
        return None

    # 검색된 문서 목록 출력
    for i, doc in enumerate(docs, 1):
        # 각 문서의 제목을 최대 60자로 제한하여 출력
        print(f"  {i}. {doc.metadata.get('title', 'N/A')[:60]}")

    # 컨텍스트 생성
    # 검색된 문서들을 하나의 문자열로 결합하여 LLM에 제공할 컨텍스트를 만듭니다
    context = "\n".join([
        f"[기사 {i+1}] {doc.metadata.get('title', '')}\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])

    # 답변 생성
    print(f"\n GPT-4o-mini 답변 생성중")
    # RAG 체인을 사용하여 컨텍스트와 질문을 바탕으로 답변 생성
    answer = rag_chain.invoke({"context": context, "question": question})


    # 최종 답변 출력
    print("\n" + "="*80)
    print(" 답변:")
    print(answer)
    print("="*80)

    # 답변 반환 (필요시 다른 곳에서 사용할 수 있도록)
    return answer

# 함수 정의 완료 메시지
print("RAG 질의응답 함수 준비 완료")

# 코드 설명:

# 1. 함수 매개변수:
#    - question: 사용자의 질문 (문자열)
#    - k: 검색할 문서 수 (기본값 3)

# 2. retriever.invoke() 호출:
#    - 멀티 벡터 검색을 수행합니다
#    - 질문과 관련된 요약 문서를 벡터 저장소에서 검색
#    - 검색된 문서의 ID로 원본 문서를 문서 저장소에서 조회
#    - 원본 문서들을 반환

# 3. 컨텍스트 생성:
#    - 리스트 컴프리헨션을 사용하여 각 문서에 번호와 제목을 추가
#    - 형식: [기사 1] 제목\n내용\n\n[기사 2] 제목\n내용...
#    - 이 컨텍스트는 LLM에 전달되어 질문에 답변하는 데 사용됩니다

# 4. RAG 체인 실행:
#    - rag_chain은 이전에 정의된 프롬프트 템플릿 → LLM → 출력 파서 체인입니다
#    - context와 question을 딕셔너리로 전달하여 답변을 생성합니다

# 5. 출력 형식:
#    - 질문 표시
#    - 검색된 문서 목록 표시
#    - 답변 생성 중 메시지
#    - 최종 답변 표시

# 6. 반환값:
#    - LLM이 생성한 답변 문자열을 반환합니다
#    - 검색 결과가 없으면 None을 반환합니다

# 주의사항 (코드 오류):
# 7번 줄에 오타가 있습니다:
#    원래: print("질문: {question}")
#    수정 필요: print(f"질문: {question}")
#    f-string을 사용하지 않으면 {question}이 그대로 문자열로 출력됩니다

# 예상 출력 형식:
# 질문: [사용자 질문]
# ================================================================================
# 검색 완료
#   1. [문서 제목 1]
#   2. [문서 제목 2]
#   3. [문서 제목 3]
#
#  GPT-4o-mini 답변 생성중
#
# ================================================================================
#  답변:
# [LLM이 생성한 답변 내용]
# ================================================================================

# RAG 시스템의 전체적인 데이터 흐름:
# 1. 사용자 질문 입력
# 2. 멀티 벡터 검색기로 관련 문서 검색
# 3. 검색된 문서들을 컨텍스트 문자열로 변환
# 4. RAG 프롬프트 템플릿에 컨텍스트와 질문 삽입
# 5. LLM이 컨텍스트를 참조하여 답변 생성
# 6. 답변 출력 및 반환

# 이 함수의 장점:
# - 사용자 친화적인 인터페이스: 각 단계를 알려주어 사용자 경험 향상
# - 투명성: 어떤 문서가 참조되었는지 표시
# - 유연성: k 값 조절로 검색 범위 조정 가능
# - 오류 처리: 검색 결과 없을 경우 적절히 처리

# 다음 단계에서는 이 함수를 실제로 호출하여 테스트할 것입니다:
# 1. "최근 경제 동향에 대해 설명해주세요" 질문 테스트
# 2. "기술 발전이 사회에 미치는 영향은?" 질문 테스트

# 실제 프로덕션 환경에서의 고려사항:
# 1. 타임아웃 설정: LLM 응답이 너무 오래 걸릴 경우 처리
# 2. 토큰 제한: 컨텍스트가 너무 길어지지 않도록 관리
# 3. 캐싱: 자주 묻는 질문의 결과 캐싱
# 4. 로깅: 질문과 답변 기록

RAG 질의응답 함수 준비 완료


In [ ]:
# RAG 테스트 1 결과 분석:
answer1 = ask_question("최근 경제 동향에 대해 설명해주세요")

# 출력 결과:
# 질문: {question}  <-- 오류: f-string이 아니므로 {question}이 그대로 출력됨
# ================================================================================
# 검색 완료
#   1. "1분기 가계 예금은 늘고 대출 증가세는 둔화"
#   2. 전자·철강·석유 업종 하반기 수출감소 불가피
#   3. 가뭄 영향에... 금배추 됐네
#
#  GPT-4o-mini 답변 생성중
#
# ================================================================================
#  답변:
# [LLM이 생성한 경제 동향 분석 답변]

# 테스트 1 결과 분석:
# 1. 검색된 문서 평가:
#    - 문서1: 가계 예금과 대출 동향 (금융 부문)
#    - 문서2: 주요 업종 수출 전망 (무역 부문)
#    - 문서3: 농산물 가격 변동 (농업 부문)
#    - 검색된 문서들은 경제 동향의 다양한 측면(금융, 무역, 농업)을 다루고 있어 적절합니다.

# 2. RAG 시스템의 작동 평가:
#    - 멀티 벡터 검색이 정상 작동: 요약문으로 검색하여 원본 문서 3개를 성공적으로 반환했습니다.
#    - 컨텍스트 구성: 검색된 문서들을 결합하여 LLM에 전달할 컨텍스트를 생성했습니다.
#    - 답변 생성: LLM이 컨텍스트를 참조하여 경제 동향에 대한 포괄적인 답변을 생성했습니다.

# 3. 생성된 답변의 품질 평가:
#    - 구조적 답변: "첫째, 둘째, 셋째"로 구분하여 체계적으로 서술했습니다.
#    - 문서 참조: 각 문장이 검색된 문서의 내용을 바탕으로 하고 있습니다.
#    - 종합 분석: 다양한 경제 지표를 종합하여 전체적인 경제 동향을 설명했습니다.
#    - RAG의 장점: LLM의 일반 지식이 아닌, 구체적인 데이터(예금 증가율, 수출 전망 등)를 기반으로 답변했습니다.

# 4. 발견된 문제점:
#    - 함수 내 오타: print("질문: {question}")에서 f-string이 누락되어 {question}이 그대로 출력되었습니다.
#    - 데이터 제한: 30개 샘플로는 경제 동향을 포괄적으로 설명하기에 부족할 수 있습니다.
#    - 문서 다양성: 더 많은 분야(고용, 물가, 부동산 등)의 문서가 포함되었다면 더 풍부한 답변이 가능했을 것입니다.

질문: {question}
검색 완료
  1. “1분기 가계 예금은 늘고 대출 증가세는 둔화”
  2. 전자·철강·석유 업종 하반기 수출감소 불가피
  3. 가뭄 영향에... 금배추 됐네

 GPT-4o-mini 답변 생성중

 답변:
최근 경제 동향은 여러 가지 측면에서 나타나고 있습니다. 

첫째, 가계의 예금이 증가하고 대출 증가세가 둔화되고 있습니다. 한국은행의 조사에 따르면, 올해 1분기 가계와 비영리단체의 순자금운용은 60조 4천억 원으로, 1년 전보다 9조 3천억 원 증가했습니다. 이는 코로나19 지원금 등으로 소득이 늘어나고, 주택매매와 같은 부동산 투자가 줄어들면서 가계의 자금 운용 규모가 확대된 것으로 분석됩니다. 그러나 주식투자 규모는 지난해 1분기보다 증가세가 둔화된 16조 원에 그쳤습니다.

둘째, 하반기 수출 전망이 부정적입니다. 원자재 가격 상승과 공급망 차질로 인해 올해 하반기 우리나라의 수출 증가세가 크게 꺾일 것으로 예상되고 있습니다. 전국경제인연합회가 실시한 조사에 따르면, 기업들은 하반기 수출이 전년 동기 대비 0.5% 증가에 그칠 것으로 전망하고 있으며, 전기전자, 철강, 석유화학 업종은 수출이 감소할 것이라고 응답했습니다. 원자재 가격 상승과 물류비 상승이 주요 요인으로 지목되고 있습니다.

셋째, 농업 부문에서는 가뭄의 영향으로 배추 가격이 급등하고 있습니다. 서울의 대형마트에서 배추 도매가격이 10kg 기준으로 1만500원에 이를 것으로 예상되며, 이는 작년 대비 90.9%, 평년 대비 39% 상승한 수치입니다. 이는 가뭄 등 날씨 영향으로 작황이 부진하고 재배면적이 감소했기 때문입니다.

종합적으로, 최근 경제 동향은 가계의 자금 운용 변화, 수출 감소 전망, 농산물 가격 상승 등 다양한 요소가 복합적으로 작용하고 있는 상황입니다.


In [ ]:
# RAG 테스트 2 결과 분석:
answer2 = ask_question("기술 발전이 사회에 미치는 영향은?")

# 출력 결과:
# 질문: {question}  <-- 동일한 오류
# ================================================================================
# 검색 완료
#   1. 윤종규 KB금융 회장 고객의 방파제 역할 수행하는 것이 금융사 핵심
#   2. 과기정통부 7월 한 달간 제11회 정보보호의 달 운영
#   3. "고객님 소비패턴 보니... 카페할인 큰 카드 알려드리죠"
#
#  GPT-4o-mini 답변 생성중
#
# ================================================================================
#  답변:
# [LLM이 생성한 기술 발전의 사회적 영향 분석 답변]

# 테스트 2 결과 분석:
# 1. 검색된 문서 평가:
#    - 문서1: 금융사의 역할 변화 (기술 발전이 금융에 미친 영향)
#    - 문서2: 정보보호 캠페인 (기술 발전에 따른 보안 중요성)
#    - 문서3: 데이터 기반 맞춤형 서비스 (소비 패턴 분석 기술)
#    - 검색된 문서들은 기술 발전의 사회적 영향을 간접적으로 보여주지만, 직접적인 기술 혁신 문서는 부족합니다.

# 2. RAG 시스템의 성능 평가:
#    - 검색 정확도: "기술 발전"이라는 추상적인 주제에 대해 관련 문서를 찾았습니다.
#    - 컨텍스트 적절성: 데이터 분석, 사이버 보안, 개인화 서비스 등 기술 발전의 사회적 영향 측면을 다룬 문서들이 포함되었습니다.
#    - 답변 적합성: LLM은 주어진 문서들을 바탕으로 기술 발전의 긍정적 영향(개인화 서비스, 보안 강화)을 설명했습니다.

# 3. 생성된 답변의 특징:
#    - 긍정적 관점: 기술 발전의 긍정적 영향에 초점을 맞춤
#    - 구체적 예시: 마이데이터 서비스, 정보보호 캠페인 등 실제 사례를 포함
#    - 사회적 관점: 기업-소비자 관계 발전과 같은 사회적 영향 강조

# 4. 시스템의 한계:
#    - 데이터셋 한계: 30개 뉴스 샘플에 기술 발전을 직접 다루는 문서가 제한적
#    - 주제 적합성: "기술 발전"은 매우 광범위한 주제인데, 데이터셋은 주로 경제 뉴스에 치중되어 있음
#    - 검색 품질: 벡터 검색이 의미적 유사도에 기반하므로, "기술"과 관련이 적은 문서도 검색될 수 있음

# 전체 시스템 평가:

# 1. 멀티 벡터 검색의 효과:
#    - 요약문 검색: 짧은 요약문으로 빠르고 정확한 검색 가능
#    - 원본문 제공: 검색된 문서의 전체 내용을 LLM에 제공하여 풍부한 컨텍스트 확보
#    - ID 매핑: doc_id를 통해 두 저장소 간 효율적 연결

# 2. RAG 시스템의 장점:
#    - 사실 기반 응답: LLM의 환각(hallucination)을 줄이고 사실에 기반한 답변 생성
#    - 최신 정보: 데이터셋의 뉴스 기사를 활용하여 최신 정보 제공
#    - 출처 투명성: 참조한 문서들을 표시하여 답변의 신뢰성 향상

# 3. 개선 필요 사항:
#    - 데이터셋 확대: 더 다양한 주제와 많은 수의 문서 필요
#    - 프롬프트 최적화: 더 정확한 답변을 위한 프롬프트 엔지니어링
#    - 검색 알고리즘 향상: 하이브리드 검색(키워드+의미) 도입 고려
#    - 코드 오류 수정: f-string 오류와 같은 기본적 문제 해결

# 4. 실제 적용 가능성:
#    - 뉴스 기반 Q&A 시스템: 최신 뉴스를 기반으로 한 질의응답 시스템 구축 가능
#    - 기업 내부 지식 관리: 내부 문서를 요약하고 원본을 저장하는 지식 관리 시스템
#    - 교육용 도구: 특정 도메인 지식을 가르치는 교육 보조 도구

# 결론:
# 멀티 벡터 RAG 시스템은 요약문으로 검색하고 원본문으로 응답하는 방식으로
# 검색 효율성과 응답 품질을 동시에 향상시킬 수 있는 유망한 접근법입니다.
# 데이터의 양과 질, 시스템의 세부 조정을 통해 더욱 향상된 성능을 기대할 수 있습니다.

질문: {question}
검색 완료
  1. 윤종규 KB금융 회장 고객의 방파제 역할 수행하는 것이 금융사 핵심
  2. 과기정통부 7월 한 달간 제11회 정보보호의 달 운영
  3. “고객님 소비패턴 보니… 카페할인 큰 카드 알려드리죠”

 GPT-4o-mini 답변 생성중

 답변:
기술 발전은 사회에 여러 가지 긍정적인 영향을 미치고 있습니다. 특히, 데이터 기반의 맞춤형 서비스 제공이 가능해짐에 따라 개인의 소비 패턴과 금융 정보를 분석하여 보다 효율적이고 개인화된 서비스를 제공할 수 있게 되었습니다. 예를 들어, 통신사의 마이데이터 서비스는 고객의 금융정보를 분석하여 최적의 금융 상품이나 서비스를 추천하는 방식으로, 소비자에게 더 나은 선택을 할 수 있도록 돕고 있습니다. 

또한, 기술 발전은 사이버 보안 분야에서도 중요한 역할을 하고 있습니다. 정보보호의 달과 같은 캠페인을 통해 사이버 위협에 대한 인식을 높이고, 정보보호 실천 문화를 확산시키는 데 기여하고 있습니다. 이는 디지털 사회에서 안전한 환경을 조성하는 데 필수적입니다.

결론적으로, 기술 발전은 개인화된 서비스 제공과 사이버 보안 강화 등 다양한 측면에서 사회에 긍정적인 영향을 미치고 있으며, 이는 기업과 소비자 간의 관계를 더욱 발전시키고 있습니다.
